# 대조학습 문장임베딩 미세조정 — MNR + 하드네거티브 + z-norm 앙상블 (Colab)

**연습 기법** (IOAI 2025 *at-home* **Chameleon** 과 동일): 사전학습 **문장 임베딩 모델**(bge·e5 계열)을
**대조학습(contrastive learning)** 으로 미세조정해 임베딩 품질을 끌어올린다. 핵심 3가지:

1. **MNR 손실**(Multiple Negatives Ranking) — 배치 안의 다른 정답쌍을 자동 음성(in-batch negative)으로 써서
   `(anchor, positive)` 만으로 강하게 학습. *positive-only CosineSimilarity 만 오래 돌리면 모든 벡터가
   한 점으로 **붕괴(collapse)*** 하는데, MNR 은 음성으로 밀어내 이를 막는다.
2. **하드 네거티브**(hard negative) — *반대 클래스인데 텍스트가 비슷해 헷갈리는* 샘플을 베이스 임베딩으로
   채굴해 `(anchor, positive, hard_neg)` 삼중쌍으로 넣는다 → 결정경계가 날카로워진다.
3. **z-norm 앙상블** — 서로 다른 두 모델(**bge-small + e5-small**, 원문제는 bge-large + e5-instruct)의
   점수를 각각 **z-정규화** 후 평균 → 척도가 다른 모델을 공정하게 합친다.

**대회**: [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) (재난 트윗 0/1 분류).
분류 헤드를 붙이는 12번(BERT) 과 달리, 여기선 **임베딩 공간에서 클래스 중심(centroid) 코사인**으로 분류한다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU. (작은 임베딩 모델이라 T4 에서 수 분 내 완료)
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.
> 💡 원래 Chameleon 은 *모델(임베딩) 제출* 형식이지만, 여기선 같은 기법을 캐글 **CSV 제출** 대회에 이식해 연습합니다.

## 0. 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "sentence-transformers", "scikit-learn", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle 에서 Disaster Tweets 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 데이터 로드 & train/val 분할
임베딩을 **미세조정(train)** 하고, **검증셋(val)** 으로 임계값과 F1 을 점검한다. `test.csv` 는 캐글 제출용.

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
test  = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
train["text"] = train["text"].fillna(""); test["text"] = test["text"].fillna("")

tr_idx, val_idx = train_test_split(np.arange(len(train)), test_size=0.15,
                                   random_state=42, stratify=train["target"].values)
tr_text  = train["text"].values[tr_idx];  tr_y  = train["target"].values[tr_idx]
val_text = train["text"].values[val_idx]; val_y = train["target"].values[val_idx]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("train:", len(tr_text), "| val:", len(val_text), "| test:", len(test), "| device:", device)

## 3. 대조학습 미세조정 — MNR 손실 + 하드 네거티브

각 모델마다 (1) **베이스 임베딩**으로 *반대 클래스 중 가장 비슷한* 하드 네거티브를 채굴하고,
(2) `(anchor, positive, hard_neg)` 삼중쌍을 **MultipleNegativesRankingLoss** 로 학습한다.
MNR 은 배치 내 다른 쌍을 음성으로 자동 사용하므로 **붕괴 없이** 같은 클래스는 모으고 다른 클래스는 밀어낸다.

In [ ]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from sklearn.neighbors import NearestNeighbors
from torch.utils.data import DataLoader

# 원문제의 bge-large + e5-instruct 를 T4 친화적인 small 변형으로 1:1 대응
MODELS = {
    "BAAI/bge-small-en-v1.5": "",          # bge: 접두사 불필요
    "intfloat/e5-small-v2":   "query: ",   # e5: 대칭 유사도엔 'query: ' 접두사 권장
}
rng = np.random.RandomState(0)

def finetune_embedder(name, prefix, epochs=1, batch_size=32):
    model = SentenceTransformer(name, device=device)
    texts = [prefix + t for t in tr_text]
    # (1) 베이스 임베딩으로 하드 네거티브 채굴
    base = model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    examples = []
    for cls in (0, 1):
        same = np.where(tr_y == cls)[0]; opp = np.where(tr_y != cls)[0]
        nn = NearestNeighbors(n_neighbors=1, metric="cosine").fit(base[opp])
        _, nbr = nn.kneighbors(base[same])
        for k, i in enumerate(same):
            j = i
            while j == i and len(same) > 1:          # 자기 자신이 아닌 같은 클래스 positive
                j = same[rng.randint(len(same))]
            hard = opp[nbr[k, 0]]                     # 반대 클래스 중 가장 비슷한 = 하드 네거티브
            examples.append(InputExample(texts=[texts[i], texts[j], texts[hard]]))
    # (2) MNR 손실로 미세조정
    loader = DataLoader(examples, shuffle=True, batch_size=batch_size)
    loss = losses.MultipleNegativesRankingLoss(model)
    model.fit(train_objectives=[(loader, loss)], epochs=epochs,
              warmup_steps=int(0.1 * len(loader)), show_progress_bar=True)
    print(f"  ✓ {name}: 삼중쌍 {len(examples)}개로 {epochs}ep 미세조정 완료")
    return model

tuned = {name: finetune_embedder(name, pfx) for name, pfx in MODELS.items()}

## 4. 임베딩 분류 점수 — 클래스 중심(centroid) 코사인
분류 헤드 없이, **train 임베딩의 클래스 중심**과의 코사인으로 *재난스러움* 점수를 낸다:
`score = cos(x, 중심_재난) − cos(x, 중심_일반)`. (Chameleon 의 코사인 유사도 비교와 동형)

In [ ]:
def disaster_score(model, prefix, texts):
    emb = model.encode([prefix + t for t in texts], batch_size=64,
                       normalize_embeddings=True, show_progress_bar=False)
    return emb

def score_split(model, prefix):
    e_tr  = disaster_score(model, prefix, tr_text)
    c1 = e_tr[tr_y == 1].mean(0); c1 /= np.linalg.norm(c1)   # 재난 중심
    c0 = e_tr[tr_y == 0].mean(0); c0 /= np.linalg.norm(c0)   # 일반 중심
    f = lambda E: E @ c1 - E @ c0
    s_tr  = f(e_tr)
    s_val = f(disaster_score(model, prefix, val_text))
    s_te  = f(disaster_score(model, prefix, test["text"].values))
    return s_tr, s_val, s_te

raw = {name: score_split(tuned[name], MODELS[name]) for name in MODELS}
print("각 모델별 (train/val/test) 점수 계산 완료")

## 5. z-norm 앙상블 & 임계값 튜닝
두 모델의 점수는 척도가 달라 그냥 더하면 한쪽이 지배한다. **train 점수의 평균·표준편차로 z-정규화**한 뒤
평균내고, **val F1** 이 최대가 되는 임계값을 고른다.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def znorm_ensemble(which):  # which: 0=train,1=val,2=test
    zs = []
    for name in MODELS:
        s_tr = raw[name][0]
        mu, sd = s_tr.mean(), s_tr.std() + 1e-8
        zs.append((raw[name][which] - mu) / sd)
    return np.mean(zs, axis=0)

z_val, z_test = znorm_ensemble(1), znorm_ensemble(2)
thr = max(np.linspace(-1.0, 1.0, 81), key=lambda t: f1_score(val_y, (z_val > t).astype(int)))
val_pred = (z_val > thr).astype(int)
print(f"best threshold = {thr:.3f}")
print(f"val accuracy = {accuracy_score(val_y, val_pred):.4f} | val F1 = {f1_score(val_y, val_pred):.4f}")

# 단일 모델 대비 앙상블 이득 확인
for name in MODELS:
    s_tr, s_val, _ = raw[name]
    z = (s_val - s_tr.mean()) / (s_tr.std() + 1e-8)
    t1 = max(np.linspace(-1, 1, 81), key=lambda t: f1_score(val_y, (z > t).astype(int)))
    print(f"  단일 {name:24s} val F1 = {f1_score(val_y, (z > t1).astype(int)):.4f}")

## 6. 캐글 제출 파일 생성 (`test.csv` → `submission.csv`)

In [ ]:
test_pred = (z_test > thr).astype(int)
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"id": test["id"], "target": test_pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path, "| 1의 비율:", round(test_pred.mean(), 3))
pd.read_csv(submission_path).head()

In [ ]:
try:
    from google.colab import files
    files.download(submission_path)
except Exception as e:
    print("자동 다운로드 불가:", e)
    print("submission.csv 위치:", submission_path)

## 🏁 제출하기
`submission.csv` 를 [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started/submit) 에 제출하세요.

**기법 요약** (IOAI 2025 Chameleon): `(anchor, positive, hard_neg)` 삼중쌍 + **MNR 손실**로 문장 임베딩을
대조학습 미세조정 → **붕괴 방지**. 서로 다른 두 임베딩 모델을 **z-norm 앙상블**로 합쳐 강건성↑.
더 끌어올리려면: epoch·batch_size 키우기(in-batch 음성 多), 하드네거티브 개수↑, 더 큰 모델(bge-base/large·e5-base),
중심 코사인 대신 미세조정 임베딩 위 **로지스틱/KNN** 헤드, val 기반 임계값 정교화.